# Hypothesis 2 — DocLayout-YOLO + EasyOCR figure extraction (test)

**Free & local.** DocLayout-YOLO finds `figure` and `figure_caption` regions on each patent sheet; EasyOCR reads the caption for the `FIG. N` label. Crops the drawing only, from the original full-res image, named `_F<label>` / `_Fu`.

> Run this with the **`doclayout_yolo`** kernel (has doclayout_yolo, easyocr, torch+CUDA, cv2). No API key, no per-image cost.

In [ ]:
import sys, importlib, time
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from config_loader import load_config
import doclayout_matcher as dm
importlib.reload(dm)

cfg = load_config()
engine = dm.build_engine()        # loads YOLOv10 + EasyOCR once (EasyOCR downloads models on first run)
model, reader, device = engine
print('Device     :', device)
print('YOLO classes:', model.names)
print('Raw images :', cfg['paths']['raw_images'])

## Pick a few sample drawing sheets
First `img*.png` sheet from the first few patent folders. Edit `samples` to target specific sheets.

In [ ]:
raw = Path(cfg['paths']['raw_images'])
samples = []
for folder in sorted(p for p in raw.iterdir() if p.is_dir()):
    sheets = sorted(folder.glob('img*.png'))
    if sheets:
        samples.append(sheets[0])
    if len(samples) >= 3:
        break
for s in samples:
    print(s.relative_to(raw))

## Run detection + show figure/caption boxes and crops

In [ ]:
out_dir = Path.cwd() / 'outputs' / 'doclayout_test'
t0 = time.time()
results = []

for img_path in samples:
    res = dm.process_image(engine, img_path, out_dir)
    results.append(res)
    labels = [c['label'] or '?' for c in res['crops']]
    print(f"\n=== {img_path.name} ===")
    print(f"  figures detected : {len(res['figures'])} | captions: {len(res['captions'])}")
    print(f"  crops saved      : {len(res['crops'])} | labels: {labels}")

    annotated = dm.draw_regions(img_path, res['figures'], res['captions'])
    plt.figure(figsize=(8, 10)); plt.imshow(annotated); plt.axis('off')
    plt.title(f"{img_path.name} — green=figure, blue=caption"); plt.show()

    n = len(res['crops'])
    if n:
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
        axes = [axes] if n == 1 else axes
        for ax, rec in zip(axes, res['crops']):
            crop_img = cv2.cvtColor(cv2.imread(str(out_dir / rec['output'])), cv2.COLOR_BGR2RGB)
            ax.imshow(crop_img); ax.axis('off')
            ax.set_title(rec['output'].split('_crop_')[-1].replace('.png', ''))
        plt.show()
    else:
        print('  (no figure regions detected)')

elapsed = time.time() - t0
print(f"\nProcessed {len(samples)} sheets in {elapsed:.1f}s  ({len(samples)/elapsed:.2f} sheets/sec)")

## Cost / throughput
This route is **free** — the only resource is GPU time. The figure below projects wall-clock time for the full dataset (first run is slower: EasyOCR model download + CUDA warmup).

In [ ]:
per = elapsed / max(1, len(samples))
N = 10839
print(f'Per-sheet (incl. first-run warmup): {per:.2f}s')
print(f'Projected {N} sheets: ~{per * N / 60:.0f} min  (steady-state will be faster)')
print('Per-image cost: €0.00')